# 03.5 Ensemble, risk-score, events, evaluation

Вход:
- `if_scores.parquet` (03.2).
- `hdb_scores.parquet` (03.3).
- `lstm_scores.parquet` (03.4).
- `annotated_v3.parquet`.

Выход:
- `model_scores_points_v3.parquet`: merged 60M точек со scores и risk_score, risk_level.
- `events_v3.parquet`: contiguous high-risk сегменты.
- `flight_risk_summary_v3.csv`: per-flight summary с event-aware ranking.
- `dashboard_flight_summary.parquet`: для Streamlit/Dash.
- `evaluation_report_v3.json`: evaluation metrics.

## Методологические решения

Шкала risk_score строится через ECDF от ensemble_score на clean calval. ensemble_score сам по себе уже усреднение percentile scores по трём моделям, а финальная шкала калибруется относительно распределения clean calval. От неё определяются три уровня риска: low (<0.90), medium (0.90-0.99), high (≥0.99). Это даёт интерпретируемую категоризацию для дашборда.

Категоризация события различает четыре случая: feature_quality_share > 0.2 идёт в отдельную категорию likely_feature_phase_artifact и не смешивается с raw DQ. Порог снижен до 0.2, изначально стоял 0.5.

Порог согласия моделей `MODEL_HIGH_THRESHOLD = 0.99` отделён от EVENT_THRESHOLD (P99 ensemble). Использовать один и тот же порог для двух разных шкал методологически нечисто.

Flight summary сортируется по ranking_score. Если у рейса есть события, берём max_event_score, иначе ensemble_p99. По ensemble_max не сортируем, потому что это даёт смещение на уровне точек.

При слиянии событий проверяется и точечный gap (≤ 5 точек), и временной (≤ 10 секунд).

Origin/destination для дашборда определяются по timestamp, а не по порядку строк, потому что порядок batch может не совпадать с порядком времени.

Evaluation_report включает ensemble и risk квантили, распределение по counts, медианы по категориям, breakdown по покрытию каждой модели.

## 1. Setup

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import os
import gc
import json
import time
import glob
from scipy.stats import spearmanr

DATA_DIR = '/content/drive/MyDrive/thesis_processed'
ARTIFACTS_DIR = os.path.join(DATA_DIR, 'models_v3_artifacts')
INPUT_ANNOTATED = os.path.join(DATA_DIR, 'european_flights_annotated_v3.parquet')

# 03.1-03.4 artifacts (inputs)
IF_SCORES = os.path.join(ARTIFACTS_DIR, 'if_scores.parquet')
HDB_SCORES = os.path.join(ARTIFACTS_DIR, 'hdb_scores.parquet')
LSTM_SCORES = os.path.join(ARTIFACTS_DIR, 'lstm_scores.parquet')
CONTRACT_PATH = os.path.join(ARTIFACTS_DIR, 'contract.json')
SPLIT_PATH = os.path.join(ARTIFACTS_DIR, 'split_metadata.json')

# 03.5 outputs
MODEL_SCORES_PATH = os.path.join(ARTIFACTS_DIR, 'model_scores_points_v3.parquet')
EVENTS_PATH = os.path.join(ARTIFACTS_DIR, 'events_v3.parquet')
FLIGHT_SUMMARY_PATH = os.path.join(ARTIFACTS_DIR, 'flight_risk_summary_v3.csv')
DASHBOARD_SUMMARY_PATH = os.path.join(ARTIFACTS_DIR, 'dashboard_flight_summary.parquet')
EVAL_REPORT_PATH = os.path.join(ARTIFACTS_DIR, 'evaluation_report_v3.json')

print('Inputs:')
for path in [IF_SCORES, HDB_SCORES, LSTM_SCORES, CONTRACT_PATH, SPLIT_PATH]:
    exists = os.path.exists(path)
    size_mb = os.path.getsize(path) / 1e6 if exists else 0
    print(f'  {os.path.basename(path):<35s}: {exists}, {size_mb:.1f} MB')

with open(CONTRACT_PATH) as f:
    contract = json.load(f)

with open(SPLIT_PATH) as f:
    split_meta = json.load(f)

DQ_HARD = contract['dq_hard_flag']
DQ_SOFT = contract['dq_soft_flag']
FEATURE_QUALITY = contract['feature_quality_flag']
RANDOM_STATE = contract['random_state']

PHASE_NAMES = {-1: 'unknown', 0: 'ground', 1: 'takeoff', 2: 'climb',
               3: 'cruise', 4: 'descent', 5: 'approach', 6: 'landing'}

# Параметры event extraction
EVENT_THRESHOLD_QUANTILE = 0.99   # P99 of ensemble_score on clean calval
MIN_EVENT_LENGTH = 10              # consecutive points
MIN_GAP_TO_MERGE = 5               # points between events to merge
MIN_TIME_GAP_TO_MERGE_SEC = 10.0   # AND time gap <= 10 seconds

# Пороги категоризации события
DQ_HARD_ARTIFACT_THRESHOLD = 0.5
FQ_FEATURE_PHASE_THRESHOLD = 0.2  # feature_quality artifacts отдельная категория
DQ_SOFT_MIXED_THRESHOLD = 0.3

# Порог согласия моделей
MODEL_HIGH_THRESHOLD = 0.99  # P99 of own model distribution

# Пороги risk_level
RISK_LEVEL_LOW_MAX = 0.90      # < 0.90 = low
RISK_LEVEL_MEDIUM_MAX = 0.99   # 0.90-0.99 = medium, >= 0.99 = high

print(f'\nEvent extraction parameters:')
print(f'  threshold quantile: P{EVENT_THRESHOLD_QUANTILE * 100:.0f}')
print(f'  min event length:   {MIN_EVENT_LENGTH} consecutive points')
print(f'  merge gap (points): {MIN_GAP_TO_MERGE}')
print(f'  merge gap (time):   {MIN_TIME_GAP_TO_MERGE_SEC}s')

print(f'\nCategorization thresholds:')
print(f'  dq_hard_share > {DQ_HARD_ARTIFACT_THRESHOLD}: likely_data_quality_artifact')
print(f'  feature_quality_share > {FQ_FEATURE_PHASE_THRESHOLD}: likely_feature_phase_artifact')
print(f'  dq_soft_share > {DQ_SOFT_MIXED_THRESHOLD}: mixed_or_derived_dynamics')
print(f'  remainder: potential_operational_anomaly')

print(f'\nModel agreement threshold: P{MODEL_HIGH_THRESHOLD * 100:.0f} '
      f'of own model percentile distribution')

print(f'\nRisk level thresholds:')
print(f'  < {RISK_LEVEL_LOW_MAX}: low')
print(f'  {RISK_LEVEL_LOW_MAX} <= x < {RISK_LEVEL_MEDIUM_MAX}: medium')
print(f'  >= {RISK_LEVEL_MEDIUM_MAX}: high')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Inputs:
  if_scores.parquet                  : True, 1363.4 MB
  hdb_scores.parquet                 : True, 428.6 MB
  lstm_scores.parquet                : True, 2712.2 MB
  contract.json                      : True, 0.0 MB
  split_metadata.json                : True, 0.3 MB

Event extraction parameters:
  threshold quantile: P99
  min event length:   10 consecutive points
  merge gap (points): 5
  merge gap (time):   10.0s

Categorization thresholds (revised):
  dq_hard_share > 0.5 → likely_data_quality_artifact
  feature_quality_share > 0.2 → likely_feature_phase_artifact
  dq_soft_share > 0.3 → mixed_or_derived_dynamics
  remainder → potential_operational_anomaly

Model agreement threshold: P99 of own model percentile distribution

Risk level thresholds:
  < 0.9 → low
  0.9 ≤ x < 0.99 → medium
  ≥ 0.99 → high


## 2. Diagnostic block (read-only sanity checks)

### 2.1 LSTM mean score quantiles

In [4]:
print('\nDiagnostic 2.1: LSTM mean score quantile check...')
t0 = time.time()

pf_lstm = pq.ParquetFile(LSTM_SCORES)
lstm_max_sample = []
lstm_mean_sample = []
SAMPLE_FRACTION = 0.05

rng = np.random.RandomState(RANDOM_STATE)
for batch in pf_lstm.iter_batches(
        batch_size=2_000_000,
        columns=['lstm_score_max_raw', 'lstm_score_mean_raw']):
    df = batch.to_pandas()
    pick = rng.random(len(df)) < SAMPLE_FRACTION
    df_sub = df[pick].dropna(subset=['lstm_score_max_raw', 'lstm_score_mean_raw'])
    if len(df_sub) > 0:
        lstm_max_sample.append(df_sub['lstm_score_max_raw'].to_numpy())
        lstm_mean_sample.append(df_sub['lstm_score_mean_raw'].to_numpy())
    del df, df_sub

lstm_max_arr = np.concatenate(lstm_max_sample)
lstm_mean_arr = np.concatenate(lstm_mean_sample)
print(f'  Sample size: max={len(lstm_max_arr):,}, mean={len(lstm_mean_arr):,}')

quantiles = [0.0, 0.5, 0.9, 0.99, 0.999, 1.0]
print(f'\n  Quantile  | max         | mean')
print(f'  ----------+-------------+-------------')
for q in quantiles:
    q_max = np.quantile(lstm_max_arr, q)
    q_mean = np.quantile(lstm_mean_arr, q)
    print(f'  P{q * 100:>7.1f} | {q_max:>11.6f} | {q_mean:>11.6f}')

ratio_p99 = np.quantile(lstm_max_arr, 0.99) / max(np.quantile(lstm_mean_arr, 0.99), 1e-12)
print(f'\n  P99 ratio (max/mean): {ratio_p99:.2f}x (expected > 1, mean softer)')

del lstm_max_sample, lstm_mean_sample, lstm_max_arr, lstm_mean_arr
print(f'Diagnostic 2.1 done: {time.time() - t0:.0f}s')


Diagnostic 2.1: LSTM mean score quantile check...
  Sample size: max=3,101,513, mean=3,101,513

  Quantile  | max         | mean
  ----------+-------------+-------------
  P    0.0 |    0.000125 |    0.000125
  P   50.0 |    0.008372 |    0.005658
  P   90.0 |    0.074384 |    0.044549
  P   99.0 |    0.620999 |    0.373788
  P   99.9 |    2.727865 |    1.813742
  P  100.0 |   33.642673 |   24.729069

  P99 ratio (max/mean): 1.66× (expected > 1, mean softer)
Diagnostic 2.1 done: 63s


### 2.2 Coverage breakdown for all 3 models

In [5]:
print('\nDiagnostic 2.2: Coverage breakdown per model x phase x DQ category...')
t0 = time.time()


def coverage_breakdown(scores_path, score_col, label):
    pf = pq.ParquetFile(scores_path)
    counts_phase = {p: {'total': 0, 'covered': 0} for p in range(-1, 7)}
    counts_split = {'calibration_val': {'total': 0, 'covered': 0},
                    'test': {'total': 0, 'covered': 0}}
    counts_dq = {'clean': {'total': 0, 'covered': 0},
                 'dq_hard': {'total': 0, 'covered': 0},
                 'dq_soft': {'total': 0, 'covered': 0},
                 'feature_quality': {'total': 0, 'covered': 0}}

    cols = ['flight_phase', 'split', DQ_HARD, DQ_SOFT, FEATURE_QUALITY, score_col]
    for batch in pf.iter_batches(batch_size=2_000_000, columns=cols):
        df = batch.to_pandas()
        covered = df[score_col].notna()

        for p in range(-1, 7):
            mask = df['flight_phase'] == p
            counts_phase[p]['total'] += int(mask.sum())
            counts_phase[p]['covered'] += int((mask & covered).sum())

        for sp in ['calibration_val', 'test']:
            mask = df['split'] == sp
            counts_split[sp]['total'] += int(mask.sum())
            counts_split[sp]['covered'] += int((mask & covered).sum())

        is_clean = (df[DQ_HARD] == 0) & (df[DQ_SOFT] == 0) & (df[FEATURE_QUALITY] == 0)
        is_dq_hard = df[DQ_HARD] == 1
        is_dq_soft = df[DQ_SOFT] == 1
        is_fq = df[FEATURE_QUALITY] == 1

        for cat_name, cat_mask in [('clean', is_clean), ('dq_hard', is_dq_hard),
                                    ('dq_soft', is_dq_soft),
                                    ('feature_quality', is_fq)]:
            counts_dq[cat_name]['total'] += int(cat_mask.sum())
            counts_dq[cat_name]['covered'] += int((cat_mask & covered).sum())

        del df

    print(f'\n  === {label} ({score_col}) ===')
    print(f'  Per split:')
    for sp, c in counts_split.items():
        pct = c['covered'] / c['total'] * 100 if c['total'] > 0 else 0
        print(f'    {sp:>16s}: {c["covered"]:,} / {c["total"]:,} ({pct:.2f}%)')
    print(f'  Per phase:')
    for p in range(-1, 7):
        c = counts_phase[p]
        if c['total'] == 0:
            continue
        pct = c['covered'] / c['total'] * 100
        flag = ''
        if c['total'] > 1000 and pct < 50:
            flag = '  LOW'
        elif c['total'] > 1000 and pct < 80:
            flag = '  partial'
        print(f'    {PHASE_NAMES[p]:>10s}: {c["covered"]:,} / {c["total"]:,} ({pct:.2f}%){flag}')
    print(f'  Per DQ category:')
    for cat, c in counts_dq.items():
        if c['total'] == 0:
            continue
        pct = c['covered'] / c['total'] * 100
        print(f'    {cat:>20s}: {c["covered"]:,} / {c["total"]:,} ({pct:.2f}%)')

    return {'phase': counts_phase, 'split': counts_split, 'dq': counts_dq}


coverage_if = coverage_breakdown(IF_SCORES, 'if_score_phase_pct', 'IF')
coverage_hdb = coverage_breakdown(HDB_SCORES, 'hdb_score_phase_pct', 'HDBSCAN')
coverage_lstm = coverage_breakdown(LSTM_SCORES, 'lstm_score_max_phase_pct', 'LSTM-AE')

print(f'\nDiagnostic 2.2 done: {time.time() - t0:.0f}s')


Diagnostic 2.2: Coverage breakdown per model × phase × DQ category...

  === IF (if_score_phase_pct) ===
  Per split:
     calibration_val:   21,390,254 /   21,390,254 (100.00%)
                test:   41,188,507 /   41,188,507 (100.00%)
  Per phase:
        ground:      142,551 /      142,551 (100.00%)
       takeoff:      165,456 /      165,456 (100.00%)
         climb:   11,850,752 /   11,850,752 (100.00%)
        cruise:   31,961,435 /   31,961,435 (100.00%)
       descent:   11,036,050 /   11,036,050 (100.00%)
      approach:    6,554,313 /    6,554,313 (100.00%)
       landing:      868,204 /      868,204 (100.00%)
  Per DQ category:
                   clean:   61,695,075 /   61,695,075 (100.00%)
                 dq_hard:      778,799 /      778,799 (100.00%)
                 dq_soft:      106,623 /      106,623 (100.00%)
         feature_quality:        6,583 /        6,583 (100.00%)

  === HDBSCAN (hdb_score_phase_pct) ===
  Per split:
     calibration_val:   21,094,477 /   21

## 3. Merge three score parquets via row_id

In [6]:
print('\nSection 3: Merge three score parquets via row_id')
t0 = time.time()

print('Reading IF scores...')
if_cols = ['row_id', 'flight_id', 'timestamp', 'flight_phase', 'split',
           DQ_HARD, DQ_SOFT, FEATURE_QUALITY, 'is_clean_reference',
           'if_score_phase_pct', 'if_score_global_pct']
if_df = pq.read_table(IF_SCORES, columns=if_cols).to_pandas()
print(f'  IF: {len(if_df):,} rows, memory {if_df.memory_usage(deep=True).sum() / 1e9:.2f} GB')

if_df['flight_phase'] = if_df['flight_phase'].astype('int8')
if_df[DQ_HARD] = if_df[DQ_HARD].astype('uint8')
if_df[DQ_SOFT] = if_df[DQ_SOFT].astype('uint8')
if_df[FEATURE_QUALITY] = if_df[FEATURE_QUALITY].astype('uint8')
if_df['is_clean_reference'] = if_df['is_clean_reference'].astype('uint8')
if_df['if_score_phase_pct'] = if_df['if_score_phase_pct'].astype('float32')
if_df['if_score_global_pct'] = if_df['if_score_global_pct'].astype('float32')
print(f'  After dtype optimization: {if_df.memory_usage(deep=True).sum() / 1e9:.2f} GB')

print('\nReading HDBSCAN scores...')
hdb_cols = ['row_id', 'hdb_score_phase_pct', 'hdb_score_global_pct']
hdb_df = pq.read_table(HDB_SCORES, columns=hdb_cols).to_pandas()
print(f'  HDB: {len(hdb_df):,} rows')
hdb_df['hdb_score_phase_pct'] = hdb_df['hdb_score_phase_pct'].astype('float32')
hdb_df['hdb_score_global_pct'] = hdb_df['hdb_score_global_pct'].astype('float32')

print('\nReading LSTM scores...')
lstm_cols = ['row_id', 'lstm_score_max_phase_pct', 'lstm_score_max_global_pct',
             'lstm_score_mean_phase_pct', 'lstm_score_count']
lstm_df = pq.read_table(LSTM_SCORES, columns=lstm_cols).to_pandas()
print(f'  LSTM: {len(lstm_df):,} rows')
lstm_df['lstm_score_max_phase_pct'] = lstm_df['lstm_score_max_phase_pct'].astype('float32')
lstm_df['lstm_score_max_global_pct'] = lstm_df['lstm_score_max_global_pct'].astype('float32')
lstm_df['lstm_score_mean_phase_pct'] = lstm_df['lstm_score_mean_phase_pct'].astype('float32')
lstm_df['lstm_score_count'] = lstm_df['lstm_score_count'].astype('int16')

print('\nMerging IF with HDBSCAN...')
merged = if_df.merge(hdb_df, on='row_id', how='left')
del if_df, hdb_df
print(f'  After IF+HDB merge: {len(merged):,} rows')

print('\nMerging with LSTM...')
merged = merged.merge(lstm_df, on='row_id', how='left')
del lstm_df
print(f'  After full merge: {len(merged):,} rows, '
      f'{merged.memory_usage(deep=True).sum() / 1e9:.2f} GB')

print(f'\nMerge complete: {time.time() - t0:.0f}s')

assert merged['row_id'].is_unique
print('Sanity: row_id unique')

print(f'\nNon-NaN coverage per model:')
for col, label in [('if_score_phase_pct', 'IF'),
                   ('hdb_score_phase_pct', 'HDBSCAN'),
                   ('lstm_score_max_phase_pct', 'LSTM')]:
    n_cov = merged[col].notna().sum()
    print(f'  {label:>10s}: {n_cov:,} ({n_cov / len(merged) * 100:.2f}%)')


=== Section 3: Merge three score parquets via row_id ===
Reading IF scores...
  IF: 62,578,761 rows, memory 2.38 GB
  After dtype optimization: 2.38 GB

Reading HDBSCAN scores...
  HDB: 63,019,803 rows

Reading LSTM scores...
  LSTM: 63,019,803 rows

Merging IF ← HDBSCAN...
  After IF+HDB merge: 62,578,761 rows

Merging ← LSTM...
  After full merge: 62,578,761 rows, 3.75 GB

Merge complete: 20s
Sanity: row_id unique ✓

Non-NaN coverage per model:
          IF:   62,578,761 (100.00%)
     HDBSCAN:   61,719,610 (98.63%)
        LSTM:   62,038,980 (99.14%)


## 4. Inter-model agreement diagnostics

In [7]:
print('\nSection 4: Inter-model agreement')
t0 = time.time()

clean_calval_mask = (
    (merged['split'] == 'calibration_val')
    & (merged['is_clean_reference'] == 1)
    & merged['if_score_phase_pct'].notna()
    & merged['hdb_score_phase_pct'].notna()
    & merged['lstm_score_max_phase_pct'].notna()
)
n_clean_calval = int(clean_calval_mask.sum())
print(f'Clean calval points with all 3 models scored: {n_clean_calval:,}')

if n_clean_calval > 0:
    sample_size = min(500_000, n_clean_calval)
    sample_idx = merged.index[clean_calval_mask].to_numpy()
    rng = np.random.RandomState(RANDOM_STATE)
    if len(sample_idx) > sample_size:
        sample_idx = rng.choice(sample_idx, size=sample_size, replace=False)

    if_s = merged.loc[sample_idx, 'if_score_phase_pct'].to_numpy()
    hdb_s = merged.loc[sample_idx, 'hdb_score_phase_pct'].to_numpy()
    lstm_s = merged.loc[sample_idx, 'lstm_score_max_phase_pct'].to_numpy()

    print(f'\nPearson correlation (n={n_clean_calval:,}):')
    if_full = merged.loc[clean_calval_mask, 'if_score_phase_pct'].to_numpy()
    hdb_full = merged.loc[clean_calval_mask, 'hdb_score_phase_pct'].to_numpy()
    lstm_full = merged.loc[clean_calval_mask, 'lstm_score_max_phase_pct'].to_numpy()
    p_if_hdb = np.corrcoef(if_full, hdb_full)[0, 1]
    p_if_lstm = np.corrcoef(if_full, lstm_full)[0, 1]
    p_hdb_lstm = np.corrcoef(hdb_full, lstm_full)[0, 1]
    print(f'  IF vs HDBSCAN:  {p_if_hdb:.2f}')
    print(f'  IF vs LSTM:     {p_if_lstm:.2f}')
    print(f'  HDBSCAN vs LSTM: {p_hdb_lstm:.2f}')

    print(f'\nSpearman correlation (n={len(sample_idx):,} sample):')
    s_if_hdb, _ = spearmanr(if_s, hdb_s)
    s_if_lstm, _ = spearmanr(if_s, lstm_s)
    s_hdb_lstm, _ = spearmanr(hdb_s, lstm_s)
    print(f'  IF vs HDBSCAN:  {s_if_hdb:.2f}')
    print(f'  IF vs LSTM:     {s_if_lstm:.2f}')
    print(f'  HDBSCAN vs LSTM: {s_hdb_lstm:.2f}')

    def jaccard_top_pct(a, b, pct=0.01):
        n = len(a)
        n_top = max(1, int(n * pct))
        if n_top >= n:
            return 1.0
        top_a = set(np.argpartition(a, -n_top)[-n_top:])
        top_b = set(np.argpartition(b, -n_top)[-n_top:])
        return len(top_a & top_b) / len(top_a | top_b)

    print(f'\nJaccard top-1% between models (n={n_clean_calval:,}):')
    j_if_hdb = jaccard_top_pct(if_full, hdb_full, 0.01)
    j_if_lstm = jaccard_top_pct(if_full, lstm_full, 0.01)
    j_hdb_lstm = jaccard_top_pct(hdb_full, lstm_full, 0.01)
    print(f'  IF vs HDBSCAN:  {j_if_hdb:.2f}')
    print(f'  IF vs LSTM:     {j_if_lstm:.2f}')
    print(f'  HDBSCAN vs LSTM: {j_hdb_lstm:.2f}')

    inter_model_agreement = {
        'n_clean_calval': n_clean_calval,
        'pearson': {
            'if_vs_hdb': float(p_if_hdb),
            'if_vs_lstm': float(p_if_lstm),
            'hdb_vs_lstm': float(p_hdb_lstm),
        },
        'spearman_sample': {
            'sample_size': int(len(sample_idx)),
            'if_vs_hdb': float(s_if_hdb),
            'if_vs_lstm': float(s_if_lstm),
            'hdb_vs_lstm': float(s_hdb_lstm),
        },
        'jaccard_top_1pct': {
            'if_vs_hdb': float(j_if_hdb),
            'if_vs_lstm': float(j_if_lstm),
            'hdb_vs_lstm': float(j_hdb_lstm),
        }
    }
    del if_full, hdb_full, lstm_full, if_s, hdb_s, lstm_s
else:
    inter_model_agreement = {'note': 'No clean calval points with all 3 scores'}

print(f'\nDiagnostic done: {time.time() - t0:.0f}s')


=== Section 4: Inter-model agreement ===
Clean calval points with all 3 models scored: 20,686,594

Pearson correlation (n=20,686,594):
  IF vs HDBSCAN:  0.4688
  IF vs LSTM:     0.3732
  HDBSCAN vs LSTM: 0.4435

Spearman correlation (n=500,000 sample):
  IF vs HDBSCAN:  0.4690
  IF vs LSTM:     0.3715
  HDBSCAN vs LSTM: 0.4522

Jaccard top-1% between models (n=20,686,594):
  IF vs HDBSCAN:  0.0766
  IF vs LSTM:     0.0635
  HDBSCAN vs LSTM: 0.1070

Diagnostic done: 9s


## 5. Ensemble fusion

In [8]:
print('\nSection 5: Ensemble fusion')
t0 = time.time()

print('Computing ensemble score (nanmean across 3 models)...')

scores_3d = np.stack([
    merged['if_score_phase_pct'].to_numpy(),
    merged['hdb_score_phase_pct'].to_numpy(),
    merged['lstm_score_max_phase_pct'].to_numpy()
], axis=1)

ensemble_score = np.nanmean(scores_3d, axis=1)
ensemble_count = (~np.isnan(scores_3d)).sum(axis=1)

merged['ensemble_score'] = ensemble_score.astype('float32')
merged['ensemble_count'] = ensemble_count.astype('uint8')

del scores_3d, ensemble_score, ensemble_count

print(f'  Done: {time.time() - t0:.0f}s')

print(f'\nEnsemble count distribution:')
ensemble_count_dist = {}
for k in [0, 1, 2, 3]:
    n = int((merged['ensemble_count'] == k).sum())
    pct = n / len(merged) * 100
    ensemble_count_dist[k] = {'n': n, 'pct': float(pct)}
    print(f'  count={k}: {n:,} ({pct:.2f}%)')

print(f'\nEnsemble score quantiles (all points):')
finite_ens = merged['ensemble_score'].dropna().to_numpy()
ensemble_quantiles = {}
for q in [0.0, 0.5, 0.9, 0.99, 0.999, 1.0]:
    v = float(np.quantile(finite_ens, q))
    ensemble_quantiles[f'p{q * 100:.1f}'] = v
    print(f'  P{q * 100:>7.1f}: {v:.2f}')

print(f'\nComputing event threshold (P99 of clean calval ensemble)...')
clean_calval_ens = merged.loc[
    (merged['split'] == 'calibration_val')
    & (merged['is_clean_reference'] == 1)
    & merged['ensemble_score'].notna(),
    'ensemble_score'
].to_numpy()
print(f'  Clean calval ensemble points: {len(clean_calval_ens):,}')
EVENT_THRESHOLD = float(np.quantile(clean_calval_ens, EVENT_THRESHOLD_QUANTILE))
print(f'  P{EVENT_THRESHOLD_QUANTILE * 100:.0f} threshold: {EVENT_THRESHOLD:.2f}')

del finite_ens


=== Section 5: Ensemble fusion ===
Computing ensemble score (nanmean across 3 models)...
  Done: 8s

Ensemble count distribution:
  count=0:            0 (0.00%)
  count=1:      166,024 (0.27%)
  count=2:    1,066,884 (1.70%)
  count=3:   61,345,853 (98.03%)

Ensemble score quantiles (all points):
  P    0.0: 0.0000
  P   50.0: 0.5422
  P   90.0: 0.8507
  P   99.0: 0.9718
  P   99.9: 0.9958
  P  100.0: 1.0000

Computing event threshold (P99 of clean calval ensemble)...
  Clean calval ensemble points: 21,083,531
  P99 threshold: 0.9684


0

## 5.5. Risk score и risk level

Финальная калиброванная шкала: ECDF percentile от ensemble_score относительно distribution на clean calval ensemble.

`risk_score` ∈ [0, 1], percentile rank.
`risk_level`:
- low: risk_score < 0.90;
- medium: 0.90 ≤ risk_score < 0.99;
- high: risk_score ≥ 0.99.

Эта шкала нужна для дашборда: ensemble_score сам по себе мало интерпретируем, а нормированный risk_score даёт сравнение точки с типичным поведением.

In [9]:
print('\nSection 5.5: Risk score + risk level (calibrated to clean calval)')
t0 = time.time()

# Reference = clean calval ensemble scores
ref_sorted = np.sort(clean_calval_ens)
print(f'  Reference (clean calval ensemble): {len(ref_sorted):,} points')
print(f'  Reference range: [{ref_sorted[0]:.2f}, {ref_sorted[-1]:.2f}]')
print(f'  Reference P50/P90/P99/P99.9: '
      f'{np.quantile(ref_sorted, [0.5, 0.9, 0.99, 0.999]).round(4)}')

# Compute risk_score via searchsorted
ens_arr = merged['ensemble_score'].to_numpy()
finite_mask = np.isfinite(ens_arr)

risk_score = np.full(len(merged), np.nan, dtype=np.float32)
risk_score[finite_mask] = (
    np.searchsorted(ref_sorted, ens_arr[finite_mask], side='right')
    / len(ref_sorted)
)
merged['risk_score'] = risk_score

# Compute risk_level
risk_level = np.full(len(merged), 'unknown', dtype=object)
risk_level[finite_mask & (risk_score < RISK_LEVEL_LOW_MAX)] = 'low'
risk_level[finite_mask & (risk_score >= RISK_LEVEL_LOW_MAX) & (risk_score < RISK_LEVEL_MEDIUM_MAX)] = 'medium'
risk_level[finite_mask & (risk_score >= RISK_LEVEL_MEDIUM_MAX)] = 'high'
merged['risk_level'] = pd.Categorical(
    risk_level, categories=['low', 'medium', 'high', 'unknown']
)
del ens_arr, finite_mask, risk_score, risk_level

# Distribution
print(f'\nRisk score quantiles:')
risk_finite = merged['risk_score'].dropna().to_numpy()
risk_quantiles = {}
for q in [0.0, 0.5, 0.9, 0.99, 0.999, 1.0]:
    v = float(np.quantile(risk_finite, q))
    risk_quantiles[f'p{q * 100:.1f}'] = v
    print(f'  P{q * 100:>7.1f}: {v:.2f}')

print(f'\nRisk level distribution:')
level_dist = {}
for level in ['low', 'medium', 'high', 'unknown']:
    n = int((merged['risk_level'] == level).sum())
    pct = n / len(merged) * 100
    level_dist[level] = {'n': n, 'pct': float(pct)}
    print(f'  {level:>10s}: {n:,} ({pct:.2f}%)')

print(f'\nRisk level distribution per split:')
for split in ['calibration_val', 'test']:
    print(f'  {split}:')
    sub = merged[merged['split'] == split]
    for level in ['low', 'medium', 'high', 'unknown']:
        n = int((sub['risk_level'] == level).sum())
        pct = n / len(sub) * 100 if len(sub) > 0 else 0
        print(f'    {level:>10s}: {n:,} ({pct:.2f}%)')

# Save clean_calval_ens stats BEFORE deletion (used in eval_report below)
clean_calval_ens_size = int(len(clean_calval_ens))
clean_calval_ens_quantiles = {
    'p0.0': float(np.quantile(clean_calval_ens, 0.0)),
    'p50.0': float(np.quantile(clean_calval_ens, 0.5)),
    'p90.0': float(np.quantile(clean_calval_ens, 0.9)),
    'p99.0': float(np.quantile(clean_calval_ens, 0.99)),
    'p99.9': float(np.quantile(clean_calval_ens, 0.999)),
    'p100.0': float(np.quantile(clean_calval_ens, 1.0)),
}

del risk_finite, clean_calval_ens, ref_sorted

print(f'Risk score + level computed: {time.time() - t0:.0f}s')


=== Section 5.5: Risk score + risk level (calibrated to clean calval) ===
  Reference (clean calval ensemble): 21,083,531 points
  Reference range: [0.0000, 1.0000]
  Reference P50/P90/P99/P99.9: [0.5318 0.8425 0.9684 0.9941]

Risk score quantiles:
  P    0.0: 0.0000
  P   50.0: 0.5160
  P   90.0: 0.9072
  P   99.0: 0.9915
  P   99.9: 0.9993
  P  100.0: 1.0000

Risk level distribution:
         low:   55,836,157 (89.23%)
      medium:    6,017,221 (9.62%)
        high:      725,383 (1.16%)
     unknown:            0 (0.00%)

Risk level distribution per split:
  calibration_val:
           low:   19,224,751 (89.88%)
        medium:    1,930,913 (9.03%)
          high:      234,590 (1.10%)
       unknown:            0 (0.00%)
  test:
           low:   36,611,406 (88.89%)
        medium:    4,086,308 (9.92%)
          high:      490,793 (1.19%)
       unknown:            0 (0.00%)
Risk score + level computed: 84s


## 6. Event extraction (with time-gap guard)

In [10]:
print('\nSection 6: Event extraction (with time-gap guard)')
t0 = time.time()

print('Sorting merged by (flight_id, timestamp)...')
merged = merged.sort_values(['flight_id', 'timestamp']).reset_index(drop=True)
print(f'  Sort done: {time.time() - t0:.0f}s')

merged['_above_thresh'] = (
    (merged['ensemble_score'] >= EVENT_THRESHOLD)
    & merged['ensemble_score'].notna()
).astype('uint8')

print('\nExtracting events per flight (with time-gap guard)...')
t1 = time.time()

events_records = []
n_flights_processed = 0

for fid, fgrp in merged.groupby('flight_id', sort=False):
    above = fgrp['_above_thresh'].to_numpy()
    timestamps = fgrp['timestamp'].to_numpy()

    changes = np.diff(np.concatenate([[0], above, [0]]))
    starts = np.where(changes == 1)[0]
    ends = np.where(changes == -1)[0] - 1

    # Merge adjacent runs separated by ≤ MIN_GAP_TO_MERGE AND ≤ MIN_TIME_GAP_TO_MERGE_SEC
    merged_runs = []
    cur_start = starts[0]
    cur_end = ends[0]
    for i in range(1, len(starts)):
        point_gap = starts[i] - cur_end - 1
        time_gap_sec = (
            (timestamps[starts[i]] - timestamps[cur_end])
            / np.timedelta64(1, 's')
        )
        if point_gap <= MIN_GAP_TO_MERGE and time_gap_sec <= MIN_TIME_GAP_TO_MERGE_SEC:
            cur_end = ends[i]
        else:
            merged_runs.append((cur_start, cur_end))
            cur_start = starts[i]
            cur_end = ends[i]
    merged_runs.append((cur_start, cur_end))

    valid_runs = [(s, e) for s, e in merged_runs if (e - s + 1) >= MIN_EVENT_LENGTH]

    if not valid_runs:
        continue

    fgrp_arr = fgrp.reset_index(drop=True)
    for s, e in valid_runs:
        run_df = fgrp_arr.iloc[s:e + 1]
        ens_scores = run_df['ensemble_score'].to_numpy()
        risk_scores = run_df['risk_score'].to_numpy()

        if_pcts = run_df['if_score_phase_pct'].to_numpy()
        hdb_pcts = run_df['hdb_score_phase_pct'].to_numpy()
        lstm_pcts = run_df['lstm_score_max_phase_pct'].to_numpy()

        phases = run_df['flight_phase'].to_numpy()
        uniq, counts = np.unique(phases, return_counts=True)
        dominant_phase = int(uniq[counts.argmax()])
        phase_diversity = len(uniq)

        events_records.append({
            'flight_id': int(fid),
            'event_start_row_id': int(run_df['row_id'].iloc[0]),
            'event_end_row_id': int(run_df['row_id'].iloc[-1]),
            'event_start_ts': run_df['timestamp'].iloc[0],
            'event_end_ts': run_df['timestamp'].iloc[-1],
            'duration_points': int(e - s + 1),
            'duration_sec': float(
                (run_df['timestamp'].iloc[-1] - run_df['timestamp'].iloc[0]).total_seconds()
            ),
            'split': run_df['split'].iloc[0],
            'dominant_phase': dominant_phase,
            'phase_name': PHASE_NAMES.get(dominant_phase, 'unknown'),
            'phase_diversity': phase_diversity,
            'ensemble_max': float(np.nanmax(ens_scores)),
            'ensemble_mean': float(np.nanmean(ens_scores)),
            'ensemble_std': float(np.nanstd(ens_scores)),
            'risk_max': float(np.nanmax(risk_scores)) if np.isfinite(risk_scores).any() else np.nan,
            'risk_mean': float(np.nanmean(risk_scores)) if np.isfinite(risk_scores).any() else np.nan,
            'if_max': float(np.nanmax(if_pcts)) if np.isfinite(if_pcts).any() else np.nan,
            'if_mean': float(np.nanmean(if_pcts)) if np.isfinite(if_pcts).any() else np.nan,
            'hdb_max': float(np.nanmax(hdb_pcts)) if np.isfinite(hdb_pcts).any() else np.nan,
            'hdb_mean': float(np.nanmean(hdb_pcts)) if np.isfinite(hdb_pcts).any() else np.nan,
            'lstm_max': float(np.nanmax(lstm_pcts)) if np.isfinite(lstm_pcts).any() else np.nan,
            'lstm_mean': float(np.nanmean(lstm_pcts)) if np.isfinite(lstm_pcts).any() else np.nan,
            'dq_hard_share': float(run_df[DQ_HARD].mean()),
            'dq_soft_share': float(run_df[DQ_SOFT].mean()),
            'feature_quality_share': float(run_df[FEATURE_QUALITY].mean()),
            'clean_point_share': float((run_df['is_clean_reference'] == 1).mean()),
            'ensemble_count_min': int(run_df['ensemble_count'].min()),
            'ensemble_count_mean': float(run_df['ensemble_count'].mean()),
        })

    n_flights_processed += 1
    if n_flights_processed % 1000 == 0:
        print(f'  Flights processed: {n_flights_processed:,}, '
              f'events so far: {len(events_records):,}', end='\r')

print()
print(f'\nEvent extraction complete: {time.time() - t1:.0f}s')
print(f'Total events: {len(events_records):,}')

merged = merged.drop(columns=['_above_thresh'])

events_df = pd.DataFrame(events_records)
del events_records

if len(events_df) > 0:
    print(f'\nEvent statistics:')
    print(f'  Total events:        {len(events_df):,}')
    print(f'  Unique flights:      {events_df["flight_id"].nunique():,}')
    print(f'  Duration (sec):')
    print(f'    P10/P50/P90/P99: '
          f'{np.quantile(events_df["duration_sec"], [0.1, 0.5, 0.9, 0.99]).round(1)}')
    print(f'  Per split:')
    for sp, n in events_df['split'].value_counts().items():
        print(f'    {sp}: {n:,}')
    print(f'  Per dominant_phase:')
    for ph, n in events_df['phase_name'].value_counts().items():
        print(f'    {ph}: {n:,}')


=== Section 6: Event extraction (with time-gap guard) ===
Sorting merged by (flight_id, timestamp)...
  Sort done: 14s

Extracting events per flight (with time-gap guard)...


Event extraction complete: 209s
Total events: 26,083

Event statistics:
  Total events:        26,083
  Unique flights:      8,420
  Duration (sec):
    P10/P50/P90/P99: [  9.   16.   58.  206.4]
  Per split:
    test: 17,613
    calibration_val: 8,470
  Per dominant_phase:
    cruise: 11,848
    climb: 5,051
    descent: 4,946
    approach: 3,076
    landing: 796
    takeoff: 224
    ground: 142


## 7. Категоризация событий

In [11]:
print('\nSection 7: Event categorization')


def categorize_event(row):
    """4-category categorization (revised - fq отдельная категория)."""
    dq_hard = row['dq_hard_share']
    dq_soft = row['dq_soft_share']
    fq = row['feature_quality_share']

    if dq_hard > DQ_HARD_ARTIFACT_THRESHOLD:
        return 'likely_data_quality_artifact'
    if fq > FQ_FEATURE_PHASE_THRESHOLD:
        return 'likely_feature_phase_artifact'
    if dq_soft > DQ_SOFT_MIXED_THRESHOLD:
        return 'mixed_or_derived_dynamics'
    return 'potential_operational_anomaly'


if len(events_df) > 0:
    events_df['category'] = events_df.apply(categorize_event, axis=1)
    print(f'\nEvent counts per category:')
    cat_counts = events_df['category'].value_counts()
    for cat, n in cat_counts.items():
        pct = n / len(events_df) * 100
        print(f'  {cat:>20s}: {n:,} ({pct:.1f}%)')

    print(f'\nCategory breakdown by split:')
    crosstab = pd.crosstab(events_df['category'], events_df['split'])
    print(crosstab)

    print(f'\nMedian event metrics by category:')
    median_metrics = events_df.groupby('category')[
        ['duration_sec', 'ensemble_max', 'ensemble_mean', 'risk_max',
         'dq_hard_share', 'dq_soft_share', 'feature_quality_share']
    ].median()
    print(median_metrics.round(3))


=== Section 7: Event categorization (revised) ===

Event counts per category:
       potential_operational_anomaly: 22,779 (87.3%)
           mixed_or_derived_dynamics:  2,205 (8.5%)
        likely_data_quality_artifact:    823 (3.2%)
       likely_feature_phase_artifact:    276 (1.1%)

Category breakdown by split:
split                          calibration_val   test
category                                             
likely_data_quality_artifact               245    578
likely_feature_phase_artifact               94    182
mixed_or_derived_dynamics                  768   1437
potential_operational_anomaly             7363  15416

Median event metrics by category:
                               duration_sec  ensemble_max  ensemble_mean  \
category                                                                   
likely_data_quality_artifact           18.0         0.995          0.985   
likely_feature_phase_artifact          17.0         0.999          0.987   
mixed_or_derived_dy

## 8. Согласие моделей на уровне события (P99 own model)

In [12]:
print('\nSection 8: Event-level model agreement')

if len(events_df) > 0:
    # Use MODEL_HIGH_THRESHOLD = 0.99 of own model percentile (not EVENT_THRESHOLD)
    events_df['if_above_thresh'] = (events_df['if_max'] >= MODEL_HIGH_THRESHOLD).astype('uint8')
    events_df['hdb_above_thresh'] = (events_df['hdb_max'] >= MODEL_HIGH_THRESHOLD).astype('uint8')
    events_df['lstm_above_thresh'] = (events_df['lstm_max'] >= MODEL_HIGH_THRESHOLD).astype('uint8')

    events_df['n_models_above_thresh'] = (
        events_df['if_above_thresh']
        + events_df['hdb_above_thresh']
        + events_df['lstm_above_thresh']
    )

    print(f'\nNumber of models reaching P{MODEL_HIGH_THRESHOLD * 100:.0f} during event:')
    n_models_dist = {}
    for k in [0, 1, 2, 3]:
        n = int((events_df['n_models_above_thresh'] == k).sum())
        pct = n / len(events_df) * 100 if len(events_df) > 0 else 0
        n_models_dist[k] = {'n': n, 'pct': float(pct)}
        print(f'  {k} models: {n:,} events ({pct:.1f}%)')

    print(f'\nWhich single model triggered when only 1 ≥ {MODEL_HIGH_THRESHOLD}?')
    single_model = events_df[events_df['n_models_above_thresh'] == 1]
    single_model_breakdown = {}
    if len(single_model) > 0:
        for model in ['if_above_thresh', 'hdb_above_thresh', 'lstm_above_thresh']:
            n = int(single_model[model].sum())
            single_model_breakdown[model] = n
            print(f'  {model:>20s}: {n:,} events')


=== Section 8: Event-level model agreement ===

Number of models reaching P99 during event:
  0 models:    821 events (3.1%)
  1 models:  6,717 events (25.8%)
  2 models: 12,076 events (46.3%)
  3 models:  6,469 events (24.8%)

Which single model triggered when only 1 ≥ 0.99?
         if_above_thresh: 1,421 events
        hdb_above_thresh: 665 events
       lstm_above_thresh: 4,631 events


## 9. Top events case studies (per category, test split)

In [13]:
print('\nSection 9: Top event case studies (TEST split)')

if len(events_df) > 0:
    test_events = events_df[events_df['split'] == 'test'].copy()
    calval_events = events_df[events_df['split'] == 'calibration_val'].copy()

    print(f'\nTest events: {len(test_events):,}')
    print(f'Calibration events: {len(calval_events):,}')

    print(f'\nTop-5 events per category (TEST split):')
    for cat in ['potential_operational_anomaly', 'mixed_or_derived_dynamics',
                'likely_feature_phase_artifact', 'likely_data_quality_artifact']:
        cat_events = test_events[test_events['category'] == cat]
        if len(cat_events) == 0:
            print(f'\n  [{cat}]: no test events')
            continue

        top5 = cat_events.nlargest(5, 'risk_max')
        print(f'\n  [{cat}] (n={len(cat_events):,} test events):')
        display_cols = ['flight_id', 'phase_name', 'duration_sec',
                        'risk_max', 'ensemble_max',
                        'if_max', 'hdb_max', 'lstm_max',
                        'dq_hard_share', 'dq_soft_share', 'feature_quality_share',
                        'n_models_above_thresh']
        print(top5[display_cols].to_string(index=False))


=== Section 9: Top event case studies (TEST split) ===

Test events: 17,613
Calibration events: 8,470

--- Top-5 events per category (TEST split) ---

  === [potential_operational_anomaly] (n=15,416 test events) ===
 flight_id phase_name  duration_sec  risk_max  ensemble_max  if_max  hdb_max  lstm_max  dq_hard_share  dq_soft_share  feature_quality_share  n_models_above_thresh
 256925247    landing           9.0       1.0      1.000000     1.0      NaN  0.998723        0.00000       0.300000                    0.0                      2
 256925318    descent         230.0       1.0      0.999997     1.0      1.0  0.999992        0.38961       0.264069                    0.0                      3
 256943208    landing          15.0       1.0      1.000000     1.0      NaN  0.998273        0.00000       0.062500                    0.0                      2
 256971353    landing          11.0       1.0      1.000000     1.0      NaN       NaN        0.00000       0.083333               

## 10. Save events to parquet

In [14]:
print('\nSection 10: Save events_v3.parquet')

if len(events_df) > 0:
    events_df['flight_id'] = events_df['flight_id'].astype('int64')
    events_df['event_start_row_id'] = events_df['event_start_row_id'].astype('int64')
    events_df['event_end_row_id'] = events_df['event_end_row_id'].astype('int64')
    events_df['duration_points'] = events_df['duration_points'].astype('int32')
    events_df['duration_sec'] = events_df['duration_sec'].astype('float32')
    events_df['dominant_phase'] = events_df['dominant_phase'].astype('int8')
    events_df['phase_diversity'] = events_df['phase_diversity'].astype('int8')
    events_df['ensemble_count_min'] = events_df['ensemble_count_min'].astype('uint8')
    for col in ['ensemble_max', 'ensemble_mean', 'ensemble_std',
                'risk_max', 'risk_mean',
                'if_max', 'if_mean', 'hdb_max', 'hdb_mean', 'lstm_max', 'lstm_mean',
                'dq_hard_share', 'dq_soft_share', 'feature_quality_share',
                'clean_point_share', 'ensemble_count_mean']:
        if col in events_df.columns:
            events_df[col] = events_df[col].astype('float32')

    events_df.to_parquet(EVENTS_PATH, index=False, compression='snappy')
    size_mb = os.path.getsize(EVENTS_PATH) / 1e6
    print(f'  Saved: {EVENTS_PATH} ({size_mb:.1f} MB, {len(events_df):,} events)')


=== Section 10: Save events_v3.parquet ===
  Saved: /content/drive/MyDrive/thesis_processed/models_v3_artifacts/events_v3.parquet (2.6 MB, 26,083 events)


## 11. Save merged scores parquet (with risk_score, risk_level)

In [16]:
print('\nSection 11: Save model_scores_points_v3.parquet')
t0 = time.time()

output_cols = ['row_id', 'flight_id', 'timestamp', 'flight_phase', 'split',
               DQ_HARD, DQ_SOFT, FEATURE_QUALITY, 'is_clean_reference',
               'if_score_phase_pct', 'if_score_global_pct',
               'hdb_score_phase_pct', 'hdb_score_global_pct',
               'lstm_score_max_phase_pct', 'lstm_score_max_global_pct',
               'lstm_score_mean_phase_pct', 'lstm_score_count',
               'ensemble_score', 'ensemble_count',
               'risk_score', 'risk_level']
output_cols = [c for c in output_cols if c in merged.columns]

print(f'Writing merged scores ({len(merged):,} rows)...')
chunk_size = 5_000_000
writer = None
for cstart in range(0, len(merged), chunk_size):
    cend = min(cstart + chunk_size, len(merged))
    chunk = merged.iloc[cstart:cend][output_cols].copy()
    table = pa.Table.from_pandas(chunk, preserve_index=False)
    if writer is None:
        writer = pq.ParquetWriter(MODEL_SCORES_PATH, table.schema,
                                   compression='snappy')
    writer.write_table(table, row_group_size=1_000_000)
    print(f'  Written {cend:,} / {len(merged):,}', end='\r')
    del chunk, table

if writer is not None:
    writer.close()
print()
size_gb = os.path.getsize(MODEL_SCORES_PATH) / 1e9
print(f'  Saved: {MODEL_SCORES_PATH} ({size_gb:.2f} GB)')
print(f'  Time: {time.time() - t0:.0f}s')


=== Section 11: Save model_scores_points_v3.parquet ===
Writing merged scores (62,578,761 rows)...
  Written 62,578,761 / 62,578,761
  Saved: /content/drive/MyDrive/thesis_processed/models_v3_artifacts/model_scores_points_v3.parquet (2.38 GB)
  Time: 63s


## 12. Flight risk summary (event-aware ranking)

In [17]:
print('\nSection 12: Flight risk summary (event-aware ranking)')
t0 = time.time()

print('Computing per-flight metrics...')

flight_agg = merged.groupby('flight_id').agg(
    split=('split', 'first'),
    n_points=('row_id', 'count'),
    duration_sec=('timestamp', lambda x: (x.max() - x.min()).total_seconds()),
    ensemble_max=('ensemble_score', 'max'),
    ensemble_mean=('ensemble_score', 'mean'),
    ensemble_p99=('ensemble_score', lambda x: np.nanquantile(x, 0.99) if x.notna().any() else np.nan),
    risk_max=('risk_score', 'max'),
    risk_p99=('risk_score', lambda x: np.nanquantile(x, 0.99) if x.notna().any() else np.nan),
    if_max=('if_score_phase_pct', 'max'),
    hdb_max=('hdb_score_phase_pct', 'max'),
    lstm_max=('lstm_score_max_phase_pct', 'max'),
    dq_hard_share=(DQ_HARD, 'mean'),
    dq_soft_share=(DQ_SOFT, 'mean'),
    feature_quality_share=(FEATURE_QUALITY, 'mean'),
    clean_point_share=('is_clean_reference', 'mean'),
).reset_index()

# Event counts per flight
if len(events_df) > 0:
    event_counts = events_df.groupby('flight_id').size().reset_index(name='n_events')
    event_max_score = events_df.groupby('flight_id')['ensemble_max'].max().reset_index(
        name='max_event_score')
    event_max_risk = events_df.groupby('flight_id')['risk_max'].max().reset_index(
        name='max_event_risk')

    # Категории
    event_categories = events_df.groupby(['flight_id', 'category']).size().unstack(
        fill_value=0).reset_index()
    event_categories.columns = ['flight_id'] + [f'n_events_{c}' for c in event_categories.columns[1:]]

    flight_agg = flight_agg.merge(event_counts, on='flight_id', how='left')
    flight_agg = flight_agg.merge(event_max_score, on='flight_id', how='left')
    flight_agg = flight_agg.merge(event_max_risk, on='flight_id', how='left')
    flight_agg = flight_agg.merge(event_categories, on='flight_id', how='left')

flight_agg['n_events'] = flight_agg.get('n_events', 0).fillna(0).astype('int32')

# Event-aware ranking_score
# Primary: max_event_risk (если есть события)
# Fallback: risk_p99 (если событий нет)
if 'max_event_risk' in flight_agg.columns:
    flight_agg['ranking_score'] = flight_agg['max_event_risk'].fillna(
        flight_agg['risk_p99']
    )
else:
    flight_agg['ranking_score'] = flight_agg['risk_p99']

# Сортировка: ranking_score, n_events, risk_p99
flight_agg = flight_agg.sort_values(
    ['ranking_score', 'n_events', 'risk_p99'],
    ascending=[False, False, False]
).reset_index(drop=True)

flight_agg.to_csv(FLIGHT_SUMMARY_PATH, index=False)
print(f'  Saved: {FLIGHT_SUMMARY_PATH} '
      f'({os.path.getsize(FLIGHT_SUMMARY_PATH) / 1e6:.1f} MB)')

print(f'\nTop-10 flights by ranking_score (event-aware):')
top_cols = ['flight_id', 'split', 'n_points', 'ranking_score',
            'max_event_risk', 'risk_p99', 'n_events',
            'dq_hard_share', 'dq_soft_share', 'feature_quality_share']
top_cols = [c for c in top_cols if c in flight_agg.columns]
top_flights = flight_agg.head(10)[top_cols]
print(top_flights.to_string(index=False))

print(f'\nFlight summary time: {time.time() - t0:.0f}s')


=== Section 12: Flight risk summary (event-aware ranking) ===
Computing per-flight metrics...
  Saved: /content/drive/MyDrive/thesis_processed/models_v3_artifacts/flight_risk_summary_v3.csv (2.5 MB)

Top-10 flights by ranking_score (event-aware):
 flight_id           split  n_points  ranking_score  max_event_risk  risk_p99  n_events  dq_hard_share  dq_soft_share  feature_quality_share
 256766264 calibration_val      6585            1.0             1.0  0.999855        17       0.010782       0.035991               0.003493
 256929330            test      2868            1.0             1.0  0.999848        15       0.156904       0.018131               0.000000
 256944338            test      3117            1.0             1.0  0.999547        10       0.034970       0.018287               0.000000
 256925318            test      4864            1.0             1.0  0.999968         8       0.080798       0.038857               0.001850
 256711692 calibration_val      7050           

## 13. Dashboard summary (origin/dest by timestamp)

In [18]:
print('\nSection 13: Dashboard flight summary (origin/dest by timestamp)')
t0 = time.time()

print('Reading annotated for lat/lon/alt aggregates...')

# Per-flight aggregator: отслеживаем origin (min ts) и dest (max ts) отдельно
flight_geo = {}
pf_anno = pq.ParquetFile(INPUT_ANNOTATED)
for batch in pf_anno.iter_batches(
        batch_size=5_000_000,
        columns=['flight_id', 'timestamp', 'latitude', 'longitude',
                 'altitude', 'groundspeed']):
    df = batch.to_pandas()
    df['timestamp'] = pd.to_datetime(df['timestamp'])

    for fid, fgrp in df.groupby('flight_id'):
        # Sort by timestamp (defensive, на случай если batch order не совпадает с ts order)
        fgrp = fgrp.sort_values('timestamp')

        lat = fgrp['latitude']
        lon = fgrp['longitude']
        alt = fgrp['altitude']
        gs = fgrp['groundspeed']
        ts = fgrp['timestamp']

        # Пропускаем строки с NaN lat/lon при выборе origin/dest
        valid_geo = lat.notna() & lon.notna()
        valid_idx = valid_geo[valid_geo].index
        first_valid = valid_idx[0]
        last_valid = valid_idx[-1]

        # Initialize or update
        if fid not in flight_geo:
            flight_geo[fid] = {
                'lat_min': float(lat.dropna().min()) if lat.notna().any() else np.nan,
                'lat_max': float(lat.dropna().max()) if lat.notna().any() else np.nan,
                'lon_min': float(lon.dropna().min()) if lon.notna().any() else np.nan,
                'lon_max': float(lon.dropna().max()) if lon.notna().any() else np.nan,
                'alt_max': float(alt.dropna().max()) if alt.notna().any() else np.nan,
                'gs_max': float(gs.dropna().max()) if gs.notna().any() else np.nan,
                'origin_ts': ts.loc[first_valid],
                'origin_lat': float(lat.loc[first_valid]),
                'origin_lon': float(lon.loc[first_valid]),
                'dest_ts': ts.loc[last_valid],
                'dest_lat': float(lat.loc[last_valid]),
                'dest_lon': float(lon.loc[last_valid]),
            }
        else:
            g = flight_geo[fid]
            if lat.notna().any():
                g['lat_min'] = min(g['lat_min'], float(lat.dropna().min()))
                g['lat_max'] = max(g['lat_max'], float(lat.dropna().max()))
                g['lon_min'] = min(g['lon_min'], float(lon.dropna().min()))
                g['lon_max'] = max(g['lon_max'], float(lon.dropna().max()))
            if alt.notna().any():
                g['alt_max'] = max(g['alt_max'], float(alt.dropna().max()))
            if gs.notna().any():
                g['gs_max'] = max(g['gs_max'], float(gs.dropna().max()))

            # Update origin if this batch has earlier valid timestamp
            if ts.loc[first_valid] < g['origin_ts']:
                g['origin_ts'] = ts.loc[first_valid]
                g['origin_lat'] = float(lat.loc[first_valid])
                g['origin_lon'] = float(lon.loc[first_valid])
            # Update dest if this batch has later valid timestamp
            if ts.loc[last_valid] > g['dest_ts']:
                g['dest_ts'] = ts.loc[last_valid]
                g['dest_lat'] = float(lat.loc[last_valid])
                g['dest_lon'] = float(lon.loc[last_valid])
    del df

geo_df = pd.DataFrame.from_dict(flight_geo, orient='index').reset_index()
geo_df.columns = ['flight_id'] + list(geo_df.columns[1:])
# Удаляем origin_ts/dest_ts (для компактности, они нужны были только для logic)
geo_df = geo_df.drop(columns=['origin_ts', 'dest_ts'], errors='ignore')
del flight_geo

print(f'  flight_geo built: {len(geo_df):,} flights')

dashboard = flight_agg.merge(geo_df, on='flight_id', how='left')
dashboard.to_parquet(DASHBOARD_SUMMARY_PATH, index=False, compression='snappy')
print(f'  Saved: {DASHBOARD_SUMMARY_PATH} '
      f'({os.path.getsize(DASHBOARD_SUMMARY_PATH) / 1e6:.1f} MB)')

del geo_df

print(f'\nDashboard summary time: {time.time() - t0:.0f}s')


=== Section 13: Dashboard flight summary (origin/dest by timestamp) ===
Reading annotated for lat/lon/alt aggregates...
  flight_geo built: 29,788 flights
  Saved: /content/drive/MyDrive/thesis_processed/models_v3_artifacts/dashboard_flight_summary.parquet (1.8 MB)

Dashboard summary time: 159s


## 14. Evaluation report (expanded)

In [19]:
print('\nSection 14: Save evaluation report (expanded)')

eval_report = {
    'event_extraction': {
        'threshold': float(EVENT_THRESHOLD),
        'threshold_quantile': EVENT_THRESHOLD_QUANTILE,
        'min_event_length': MIN_EVENT_LENGTH,
        'min_gap_to_merge_points': MIN_GAP_TO_MERGE,
        'min_gap_to_merge_sec': MIN_TIME_GAP_TO_MERGE_SEC,
        'total_events': int(len(events_df)),
        'unique_flights_with_events': int(events_df['flight_id'].nunique())
                                       if len(events_df) > 0 else 0,
    },
    'risk_score': {
        'reference_distribution': 'clean calibration_val ensemble_score',
        'reference_size': clean_calval_ens_size,
        'reference_quantiles': clean_calval_ens_quantiles,
        'risk_quantiles': risk_quantiles,
        'risk_level_thresholds': {
            'low_max': RISK_LEVEL_LOW_MAX,
            'medium_max': RISK_LEVEL_MEDIUM_MAX,
        },
        'risk_level_distribution_all': level_dist,
    },
    'ensemble': {
        'count_distribution': ensemble_count_dist,
        'score_quantiles': ensemble_quantiles,
    },
    'inter_model_agreement': inter_model_agreement,
    'categorization': {
        'thresholds': {
            'dq_hard_artifact': DQ_HARD_ARTIFACT_THRESHOLD,
            'feature_quality_phase_artifact': FQ_FEATURE_PHASE_THRESHOLD,
            'dq_soft_mixed': DQ_SOFT_MIXED_THRESHOLD,
        },
        'counts': events_df['category'].value_counts().to_dict()
                  if len(events_df) > 0 else {},
        'median_metrics_by_category': (
            events_df.groupby('category')[
                ['duration_sec', 'ensemble_max', 'risk_max',
                 'dq_hard_share', 'dq_soft_share', 'feature_quality_share']
            ].median().round(4).to_dict()
            if len(events_df) > 0 else {}
        ),
    },
    'model_agreement_events': {
        'high_threshold': MODEL_HIGH_THRESHOLD,
        'n_models_above_distribution': n_models_dist if len(events_df) > 0 else {},
        'single_model_breakdown': single_model_breakdown if len(events_df) > 0 else {},
    },
    'events_per_split': events_df['split'].value_counts().to_dict()
                        if len(events_df) > 0 else {},
    'events_per_phase': events_df['phase_name'].value_counts().to_dict()
                        if len(events_df) > 0 else {},
    'coverage_per_model': {
        'if_total_coverage': float(merged['if_score_phase_pct'].notna().mean()),
        'hdb_total_coverage': float(merged['hdb_score_phase_pct'].notna().mean()),
        'lstm_total_coverage': float(merged['lstm_score_max_phase_pct'].notna().mean()),
    },
}


def to_serializable(obj):
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, dict):
        return {str(k): to_serializable(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [to_serializable(v) for v in obj]
    return obj


with open(EVAL_REPORT_PATH, 'w') as f:
    json.dump(to_serializable(eval_report), f, indent=2)
print(f'  Saved: {EVAL_REPORT_PATH}')


=== Section 14: Save evaluation report (expanded) ===
  Saved: /content/drive/MyDrive/thesis_processed/models_v3_artifacts/evaluation_report_v3.json


## 15. Final summary

In [20]:
print('\n' + '=' * 60)
print('03.5 FINAL SUMMARY')
print('=' * 60)

print(f'\nMerged scores: {len(merged):,} points')
print(f'  IF coverage:   {merged["if_score_phase_pct"].notna().sum() / len(merged) * 100:.2f}%')
print(f'  HDB coverage:  {merged["hdb_score_phase_pct"].notna().sum() / len(merged) * 100:.2f}%')
print(f'  LSTM coverage: {merged["lstm_score_max_phase_pct"].notna().sum() / len(merged) * 100:.2f}%')

print(f'\nEnsemble:')
print(f'  Threshold (P{EVENT_THRESHOLD_QUANTILE * 100:.0f} on clean calval): '
      f'{EVENT_THRESHOLD:.2f}')

print(f'\nRisk levels:')
for level in ['low', 'medium', 'high']:
    print(f'  {level:>8s}: {level_dist[level]["n"]:,} ({level_dist[level]["pct"]:.2f}%)')

print(f'\nEvents:')
print(f'  Total: {len(events_df):,}')
if len(events_df) > 0:
    print(f'  Per category:')
    for cat, n in events_df['category'].value_counts().items():
        print(f'    {cat}: {n:,}')

print(f'\nInter-model agreement (clean calval, Jaccard top-1%):')
if 'jaccard_top_1pct' in inter_model_agreement:
    j = inter_model_agreement['jaccard_top_1pct']
    print(f'  IF vs HDBSCAN:   {j["if_vs_hdb"]:.2f}')
    print(f'  IF vs LSTM:      {j["if_vs_lstm"]:.2f}')
    print(f'  HDBSCAN vs LSTM: {j["hdb_vs_lstm"]:.2f}')

print(f'\nArtifacts:')
for path in [MODEL_SCORES_PATH, EVENTS_PATH, FLIGHT_SUMMARY_PATH,
             DASHBOARD_SUMMARY_PATH, EVAL_REPORT_PATH]:
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1e6
        print(f'  OK   {os.path.basename(path):<40s} ({size_mb:>7.1f} MB)')

print('\n03.5 complete. ML pipeline finished. Ready for thesis writing + dashboard.')


03.5 FINAL SUMMARY

Merged scores: 62,578,761 points
  IF coverage:   100.00%
  HDB coverage:  98.63%
  LSTM coverage: 99.14%

Ensemble:
  Threshold (P99 on clean calval): 0.9684

Risk levels:
       low:   55,836,157 (89.23%)
    medium:    6,017,221 (9.62%)
      high:      725,383 (1.16%)

Events:
  Total: 26,083
  Per category:
    potential_operational_anomaly: 22,779
    mixed_or_derived_dynamics: 2,205
    likely_data_quality_artifact: 823
    likely_feature_phase_artifact: 276

Inter-model agreement (clean calval, Jaccard top-1%):
  IF vs HDBSCAN:   0.0766
  IF vs LSTM:      0.0635
  HDBSCAN vs LSTM: 0.1070

Artifacts:
  ✓ model_scores_points_v3.parquet           ( 2377.9 MB)
  ✓ events_v3.parquet                        (    2.6 MB)
  ✓ flight_risk_summary_v3.csv               (    2.5 MB)
  ✓ dashboard_flight_summary.parquet         (    1.8 MB)
  ✓ evaluation_report_v3.json                (    0.0 MB)

03.5 complete. ML pipeline finished. Ready for thesis writing + dashboard